# 12 · Report figures (CardioSafe)
Reads what notebooks 02–11 already saved and makes every figure/table for the report. **No training.** Run all cells; at the end a zip downloads. Each section is independent, so one failure won't stop the rest.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, warnings, zipfile, traceback
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, confusion_matrix
warnings.filterwarnings('ignore')
WORK = '/content/drive/MyDrive/CardioSafe'
FIG = f'{WORK}/report_figures'; os.makedirs(FIG, exist_ok=True)
sys.path.insert(0, WORK)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25})
CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
COL = ['#2a6fdb', '#d6453d', '#2b9a66', '#e0952a', '#8a5cc2']
LOG = []
def save(name):
    p = f'{FIG}/{name}.png'; plt.tight_layout(); plt.savefig(p, bbox_inches='tight'); plt.close(); LOG.append(name); print('saved', name)
def section(fn):
    def w():
        try: fn()
        except Exception as e:
            print(f'!! {fn.__name__} failed: {type(e).__name__}: {e}'); traceback.print_exc(limit=2)
    return w
print('files in CardioSafe:', sorted(os.listdir(WORK))[:60])
print('preds:', sorted(os.listdir(f'{WORK}/preds'))[:40] if os.path.isdir(f'{WORK}/preds') else 'NO preds folder')

## 1 · Load test labels and ensemble probabilities

In [ ]:
def load_preds(path):
    z = np.load(path, allow_pickle=True); return {k: z[k] for k in z.files}
pf = sorted(glob.glob(f'{WORK}/preds/*.npz'))
print(len(pf), 'pred files; example keys:', {os.path.basename(pf[0]): {k: v.shape for k, v in load_preds(pf[0]).items()}} if pf else None)

def pick_test(d):
    """(probs[N,5], labels[N,5] or None) from a preds npz, whatever the key names are."""
    P = Y = None
    for k, v in d.items():
        kl = k.lower()
        if v.ndim == 2 and v.shape[1] == 5 and 'test' in kl:
            if set(np.unique(v)) <= {0, 1, 0., 1.} and ('y' in kl or 'label' in kl or 'true' in kl): Y = v
            elif P is None or kl.startswith('p'): P = v
    return P, Y

members = json.load(open(f'{WORK}/trust_config.json'))['members']
probs, Ytest = {}, None
for m in members:
    cand = [f for f in pf if os.path.basename(f).startswith(m + '.') or os.path.basename(f) == f'{m}.npz']
    if not cand: print('no preds for', m); continue
    P, Y = pick_test(load_preds(cand[0]))
    if P is not None: probs[m] = P
    if Y is not None and Ytest is None: Ytest = Y
print('members with test probs:', len(probs), '/', len(members))

if Ytest is None:                                   # fall back to labels stored with the data helpers
    for f in pf:
        d = load_preds(f)
        for k, v in d.items():
            if v.ndim == 2 and v.shape[1] == 5 and set(np.unique(v)) <= {0, 1, 0., 1.}: Ytest = v; break
        if Ytest is not None: break
if Ytest is None:
    import cardiosafe_common as C, inspect
    print('load_data signature:', inspect.signature(C.load_data))
    out = C.load_data(); items = out if isinstance(out, (tuple, list)) else list(out.values()) if isinstance(out, dict) else [out]
    for it in items:
        a = np.asarray(it) if not isinstance(it, (str, dict)) else None
        if a is not None and a.ndim == 2 and a.shape[1] == 5 and set(np.unique(a)) <= {0, 1, 0., 1.} and len(a) == len(next(iter(probs.values()))): Ytest = a
print('labels:', None if Ytest is None else Ytest.shape)
art = np.load(f'{WORK}/trust_artifacts.npz'); Tcal = float(art['T']); ths = np.asarray(art['thresholds'])
raw = np.mean([probs[m] for m in probs], 0)                      # ensemble mean (as stored)
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
cal = 1 / (1 + np.exp(-logit(raw) / Tcal))                       # temperature-scaled
assert Ytest is not None and len(Ytest) == len(raw), 'labels missing or length mismatch'
print('test records:', len(Ytest), '| T =', round(Tcal, 3), '| prevalence:', dict(zip(CLASSES, Ytest.mean(0).round(3))))

## 2 · ROC, PR, confusion matrices, calibration (ensemble)

In [ ]:
@section
def roc_pr():
    fig, ax = plt.subplots(figsize=(6, 5))
    rows = []
    for k, c in enumerate(CLASSES):
        f, t, _ = roc_curve(Ytest[:, k], cal[:, k]); a = auc(f, t); ax.plot(f, t, color=COL[k], lw=2, label=f'{c} (AUC {a:.3f})'); rows.append((c, a))
    ax.plot([0, 1], [0, 1], 'k--', lw=1); ax.set(xlabel='False positive rate', ylabel='True positive rate', title='ROC curves, 15-model ensemble (PTB-XL fold 10)'); ax.legend(loc='lower right'); save('fig_roc_ensemble')
    fig, ax = plt.subplots(figsize=(6, 5)); pr = []
    for k, c in enumerate(CLASSES):
        p, r, _ = precision_recall_curve(Ytest[:, k], cal[:, k]); ap = average_precision_score(Ytest[:, k], cal[:, k]); ax.plot(r, p, color=COL[k], lw=2, label=f'{c} (AP {ap:.3f})'); pr.append((c, ap))
    ax.set(xlabel='Recall', ylabel='Precision', title='Precision-recall curves, 15-model ensemble'); ax.legend(loc='lower left'); save('fig_pr_ensemble')
    pd.DataFrame({'class': CLASSES, 'AUROC': [a for _, a in rows], 'AUPRC': [a for _, a in pr], 'prevalence': Ytest.mean(0), 'threshold': ths}).round(4).to_csv(f'{FIG}/table_ensemble_per_class.csv', index=False)
roc_pr()

@section
def confusion():
    fig, axs = plt.subplots(1, 5, figsize=(15, 3.2)); rows = []
    for k, c in enumerate(CLASSES):
        pred = (cal[:, k] >= ths[k]).astype(int); cm = confusion_matrix(Ytest[:, k], pred, labels=[0, 1]); tn, fp, fn, tp = cm.ravel()
        axs[k].imshow(cm, cmap='Blues'); axs[k].grid(False)
        for i in range(2):
            for j in range(2): axs[k].text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=12)
        axs[k].set(title=f'{c} (thr {ths[k]:.2f})', xticks=[0, 1], yticks=[0, 1], xticklabels=['neg', 'pos'], yticklabels=['neg', 'pos'], xlabel='Predicted', ylabel='Actual')
        rows.append({'class': c, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn, 'sensitivity': tp / max(tp + fn, 1), 'specificity': tn / max(tn + fp, 1), 'precision': tp / max(tp + fp, 1)})
    save('fig_confusion_per_class'); pd.DataFrame(rows).round(4).to_csv(f'{FIG}/table_confusion.csv', index=False)
confusion()

@section
def reliability():
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
    for ax, P, ttl in ((axs[0], raw, 'Before temperature scaling'), (axs[1], cal, f'After temperature scaling (T={Tcal:.2f})')):
        for k, c in enumerate(CLASSES):
            bins = np.quantile(P[:, k], np.linspace(0, 1, 11)); idx = np.clip(np.digitize(P[:, k], bins[1:-1]), 0, 9)
            xs = [P[idx == b, k].mean() for b in range(10) if (idx == b).sum()]; ys = [Ytest[idx == b, k].mean() for b in range(10) if (idx == b).sum()]
            ax.plot(xs, ys, 'o-', color=COL[k], ms=3, label=c)
        ax.plot([0, 1], [0, 1], 'k--', lw=1); ax.set(title=ttl, xlabel='Predicted probability')
    axs[0].set_ylabel('Observed frequency'); axs[0].legend(); save('fig_reliability')
reliability()

@section
def risk_coverage():
    P = np.stack([probs[m] for m in probs]); sd = P.std(0).mean(1)           # ensemble disagreement per record
    pred = (cal >= ths).astype(int); err = (pred != Ytest).any(1)
    order = np.argsort(sd); cov = np.arange(1, len(sd) + 1) / len(sd); risk = np.cumsum(err[order]) / np.arange(1, len(sd) + 1)
    fig, ax = plt.subplots(figsize=(6, 4.2)); ax.plot(cov, risk, lw=2, color=COL[0]); ax.axhline(err.mean(), color='k', ls='--', lw=1, label=f'no rejection ({err.mean():.2f})')
    ax.set(xlabel='Coverage (fraction of records answered)', ylabel='Records with any wrong label', title='Risk-coverage using ensemble disagreement'); ax.legend(); save('fig_risk_coverage')
risk_coverage()

## 3 · Model comparison and saved result tables

In [ ]:
R = lambda n: pd.read_csv(f'{WORK}/{n}')
@section
def comparison():
    d = R('results_03_final_comparison.csv'); d = d.rename(columns={d.columns[0]: 'model'}) if 'model' not in d.columns else d
    d = d.sort_values('AUROC'); fig, ax = plt.subplots(figsize=(7.5, 5))
    err = np.vstack([d.AUROC - d.AUROC_lo, d.AUROC_hi - d.AUROC]) if 'AUROC_lo' in d else None
    ax.barh(d.model, d.AUROC, xerr=err, color=[COL[0] if 'ensemble' in m else '#9aa5b1' for m in d.model], capsize=2); ax.set_xlim(0.8, 0.96)
    for i, v in enumerate(d.AUROC): ax.text(v + .002, i, f'{v:.3f}', va='center', fontsize=8)
    ax.set(xlabel='Macro AUROC, test fold 10 (95% CI)', title='Model comparison'); save('fig_model_comparison')
    cols = [c for c in d.columns if c.startswith('AUROC_') and c not in ('AUROC_lo', 'AUROC_hi')]
    if cols:
        fig, ax = plt.subplots(figsize=(7, 5)); im = ax.imshow(d[cols].values, cmap='viridis', aspect='auto'); ax.grid(False)
        ax.set(xticks=range(len(cols)), xticklabels=[c[6:] for c in cols], yticks=range(len(d)), yticklabels=d.model)
        for i in range(len(d)):
            for j in range(len(cols)): ax.text(j, i, f'{d[cols].values[i, j]:.2f}', ha='center', va='center', color='w', fontsize=7)
        ax.set_title('Per-class AUROC by model'); save('fig_perclass_heatmap')
comparison()

@section
def seed_spread():
    d = R('results_03_runs.csv'); d['arch'] = d['exp'] if 'exp' in d else d.iloc[:, 0].str.rsplit('_s', n=1).str[0]
    fig, ax = plt.subplots(figsize=(7, 4)); g = d.groupby('arch').AUROC
    for i, (a, v) in enumerate(g): ax.scatter([i] * len(v), v, color=COL[0], s=30); 
    ax.set_xticks(range(g.ngroups)); ax.set_xticklabels([a for a, _ in g], rotation=30, ha='right'); ax.set(ylabel='Test AUROC', title='Seed-to-seed spread per architecture'); save('fig_seed_spread')
seed_spread()

@section
def conformal_and_others():
    for n in ['results_04_calibration.csv', 'results_04_conformal.csv', 'results_04_uncertainty_auroc.csv', 'results_04_uncertainty_error.csv', 'results_05_faithfulness.csv', 'results_06b_benchmarks.csv', 'results_06b_flag.csv', 'results_06b_three_way.csv', 'results_08_ocr.csv', 'results_09_scenarios.csv']:
        if os.path.exists(f'{WORK}/{n}'):
            import shutil; shutil.copy(f'{WORK}/{n}', f'{FIG}/table_{n}'); print('\n###', n); print(R(n).round(3).to_string(index=False)[:1800])
    if os.path.exists(f'{WORK}/results_08_ocr.csv'):
        d = R('results_08_ocr.csv'); num = d.select_dtypes('number').columns.tolist(); lab = d.columns[0]
        if num:
            d.set_index(lab)[num[:4]].plot.bar(figsize=(8, 4), rot=30); plt.title('Prescription OCR benchmark'); plt.ylabel('score'); save('fig_ocr_benchmark')
conformal_and_others()

## 4 · QTc estimator (scatter, Bland-Altman)

In [ ]:
@section
def qtc():
    d = pd.read_csv(f'{WORK}/qtc_test_predictions.csv'); print('columns:', list(d.columns))
    lc = {c.lower(): c for c in d.columns}
    def find(*keys, avoid=()):
        for c in d.columns:
            cl = c.lower()
            if all(k in cl for k in keys) and not any(a in cl for a in avoid): return c
    ref = find('qt', avoid=('pred', 'est', 'rr', 'sd')) if find('qtc', 'true') is None and find('qtc', 'ref') is None else (find('qtc', 'true') or find('qtc', 'ref'))
    for k in (('qt', 'true'), ('qt', 'ref'), ('qt', 'gt'), ('qt', 'target')):
        c = find(*k, avoid=('rr', 'hr'))
        if c: ref = c; break
    prd = None
    for k in (('qt', 'pred'), ('qt', 'est'), ('qt', 'hat')):
        c = find(*k, avoid=('rr', 'hr', 'sd'))
        if c: prd = c; break
    print('using ref =', ref, '| pred =', prd)
    x, y = d[ref].values.astype(float), d[prd].values.astype(float); m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    r = np.corrcoef(x, y)[0, 1]; diff = y - x; bias, sdv = diff.mean(), diff.std()
    fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
    axs[0].scatter(x, y, s=4, alpha=.35, color=COL[0]); lo, hi = min(x.min(), y.min()), max(x.max(), y.max()); axs[0].plot([lo, hi], [lo, hi], 'k--', lw=1)
    axs[0].set(xlabel=f'Reference ({ref})', ylabel=f'Model ({prd})', title=f'Agreement, r = {r:.2f}, MAE = {np.abs(diff).mean():.1f}')
    axs[1].scatter((x + y) / 2, diff, s=4, alpha=.35, color=COL[0]); axs[1].axhline(bias, color='k'); axs[1].axhline(bias + 1.96 * sdv, color='r', ls='--'); axs[1].axhline(bias - 1.96 * sdv, color='r', ls='--')
    axs[1].set(xlabel='Mean of reference and model', ylabel='Model - reference', title=f'Bland-Altman: bias {bias:.1f}, limits {bias - 1.96 * sdv:.1f} to {bias + 1.96 * sdv:.1f}'); save('fig_qtc_agreement')
qtc()

## 5 · Copy the figures made by earlier notebooks, then zip

In [ ]:
import shutil
for f in glob.glob(f'{WORK}/*.png') + glob.glob(f'{WORK}/figures/*.png') + glob.glob(f'{WORK}/figs/*.png'):
    shutil.copy(f, f'{FIG}/earlier_{os.path.basename(f)}')
for n in ['FINAL_METRICS.md', 'trust_config.json', 'qtc_config_learned.json']:
    if os.path.exists(f'{WORK}/{n}'): shutil.copy(f'{WORK}/{n}', f'{FIG}/{n}')
zp = '/content/report_figures.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{FIG}/*')): z.write(f, os.path.basename(f))
print('\nmade this run:', LOG); print('in zip:', len(os.listdir(FIG)), 'files')
shutil.copy(zp, f'{WORK}/report_figures.zip')
try:
    from google.colab import files; files.download(zp)
except Exception as e: print('download manually from MyDrive/CardioSafe/report_figures.zip', e)